# Studying the final neutron star population

In [ ]:
import astropy.coordinates as coord
import astropy.units as u
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy.optimize import curve_fit
from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import pypopsyn.simulator.multiband_emission.emission_radio as er

import utilities.plot_settings

In [ ]:
def linear_relation(x: np.ndarray, a: float, b:float) -> np.ndarray:
    """
    A first degree linear equation.
    
    Args:
        x (np.ndarray): input array of values.
        a (float): slope of the linear relation.
        b (float): intercept of the linear relation.
        
    Returns:
        (np.ndarray): output values of the linear relation.
    """
    
    y = a*x + b
    
    return y

Select a `final_population.pkl.gz` file to import:

In [ ]:
data = pd.read_pickle("../examples/data/simulation_maxwell_sigma265_h018/final_population.pkl.gz", compression="gzip")
data.head()

In [ ]:
x = data["x"]["[kpc]"].to_numpy()
y = data["y"]["[kpc]"].to_numpy()
z = data["z"]["[kpc]"].to_numpy()
RA = data["RA"]["[deg]"].to_numpy()
DEC = data["DEC"]["[deg]"].to_numpy()
pm_RA = data["pm_RA"]["[mas yr^-1]"].to_numpy()
pm_DEC = data["pm_DEC"]["[mas yr^-1]"].to_numpy()
v_r = data["v_r"]["[km s^-1]"].to_numpy()
v_phi = data["v_phi"]["[km s^-1]"].to_numpy()
v_z = data["v_z"]["[km s^-1]"].to_numpy()
dist = data["d"]["[kpc]"].to_numpy()
B = data["B"]["[G]"].to_numpy()
chi = data["chi"]["[rad]"].to_numpy()
P = data["P"]["[s]"].to_numpy()
P_dot = data["P_dot"]["[s yr^-1]"].to_numpy()
L_radio = data["L_radio"]["[erg s^-1 Hz^-1]"].to_numpy()
S_radio_Jy = data["S_radio"]["[Jy]"].to_numpy()
w_int = data["w_int"]["[s]"].to_numpy()
intercepted = data["intercepted"][" "].to_numpy(dtype=bool)
detected = data["detected"][" "].to_numpy(dtype=bool)
age = data["age"]["[yr]"].to_numpy()

print(len(intercepted[intercepted==True])/len(intercepted))
print(len(detected[detected==True])/len(detected))

## Positional information

Top view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.2,
    rasterized=True
)

ax.plot(
    x[intercepted],
    y[intercepted],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=0.2,
    rasterized=True
)
ax.plot(
    x[detected],
    y[detected],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.,
    rasterized=True
)

ax.plot(0.0, 8.3, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Side view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.2,
    rasterized=True,
)

ax.plot(
    x[intercepted],
    z[intercepted],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=0.2,
    rasterized=True,
)
ax.plot(
    x[detected],
    z[detected],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.,
    rasterized=True
)


ax.plot(0.0, 0.02, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Histrogramming the pulsars radial position and comparing to underlying initial position PDF

In [ ]:
def pdf_r(r: float) -> float:
    """
    The Milky Way's stellar radial density in the galactic plane according
    to eq. (15) of Yusifov & Küçük (2004).

    Args:
        r (float): distance from the galactic center in [kpc].

    Returns:
        float: stellar radial density in [1/kpc].
    """

    # Here we keep R_sun = 8.5 kpc for consistency with the results
    # of Yusifov & Küçük (2004)
    rsun = 8.5  # Sun's distance from the galactic center in [kpc].
    A = 37.6  # +- 1.90 [1/kpc^2]
    a = 1.64  # +-0.11
    b = 4.01  # +-0.24
    r1 = 0.55  # +- 0.10 [kpc]

    # Stellar surface density following eq. (15) of Yusifov & Küçük (2004).
    rho = (
        A
        * ((r + r1) / (rsun + r1)) ** a
        * np.exp(-b * (r - rsun) / (rsun + r1))
    )

    # Multiply the stellar surface density with the area element in polar coordinates.
    pdf_r = 2 * np.pi * r * rho

    return pdf_r

For normalization purposes, determine the area underneath the theoretical PDF curve:

In [ ]:
pdf_area = quad(pdf_r, 0, 100)[0]
print(pdf_area)

In [ ]:
r = np.sqrt(x**2 + y**2)
r_bins = np.linspace(0.0, 30.0, 31)

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    r,
    bins=r_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Simulated all"
)
ax.hist(
    r[intercepted],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="intercepting our LOS",
)
ax.hist(
    r[detected],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="detected by PMPS",
)
ax.plot(
    r_bins,
    pdf_r(r_bins) / pdf_area * 1.e5,
    linestyle="--",
    lw=4,
    color="black",
    alpha=1,
    label="Initial YK04",
)
plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"Number of NSs")
plt.xlim(0.0, 30.0)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Histrogramming the pulsars $z$ position and comparing to underlying PDF

In [ ]:
def pdf_z(z: float) -> float:
    """
    Probability density function for the height from the galactic equatorial plane
    according to eq. (2) in Gullon et al. (2014).

    Args:
        z (float): distance from the galactic plane in [kpc].

    Returns:
        float: distribution of stars per kpc in z direction.
    """

    # We use an exponential distribution as given by Wainscoat et al. (1992)
    # and choose a mean scale height characteristic for a young distribution as
    # obtained by Gullon et al. (2014).

    h_c = 0.18
    pdf_z = 1.0 / h_c * np.exp(-z / h_c)

    return pdf_z

In [ ]:
pdf_area = quad(pdf_z, 0, 5)[0]
print(pdf_area)

In [ ]:
z_bins = np.linspace(0.0, 5.0, 31)

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    z,
    bins=z_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Simulated all",
)
ax.hist(
    z[intercepted],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="intercepting our LOS",
)
ax.hist(
    z[detected],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="detected by PMPS",
)
ax.plot(
    z_bins,
    pdf_z(z_bins) * 1.e4,
    linestyle="--",
    lw=4,
    color="black",
    alpha=1,
    label="Theoretical exp",
)
plt.xlabel(r"$z$ [kpc]")
plt.ylabel(r"Number of NS")
plt.xlim(0.0, 5.0)
plt.ylim(0.1, 1.e6)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

## Proper velocity comparison

In [ ]:
fig, ax = plt.subplots()
x_bins = np.linspace(-1500.,1500.,51)  

ax.hist(
    v_r,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"$v_r$",
)
ax.hist(
    v_phi,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"$v_{\phi}$",
)
ax.hist(
    v_z,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"$v_{z}$",
)
ax.set_xlabel(r"Velocity components [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0, fontsize=20)

plt.show()

Distribution of the total velocity magnitude

In [ ]:
def pdf_kick_velocity_maxwell(v: float) -> float:
    """
    Maxwell probability density function for the neutron stars' initial kick
    velocity magnitude following Hobbs et al. (2005).

    Args:
        v (float): initial kick velocity magnitude in [km/s].

    Returns:
        float: stellar kick velocity distribution in [1/(km/s)].
    """
    sigma = 265.
    pdf_vk = (
        np.sqrt(2 / np.pi)
        * v ** 2
        / (sigma ** 3)
        * np.exp(-(v ** 2) / (2 * sigma ** 2))
    )

    return pdf_vk

In [ ]:
v_tot = np.sqrt(v_r**2 + v_phi**2 + v_z**2)

fig, ax = plt.subplots(figsize=(15,8))
v_bins = np.linspace(0,1500.,31)  

ax.hist(
    v_tot,
    bins=v_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    v_tot[intercepted],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"intercepting our LOS",
)
ax.hist(
    v_tot[detected],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"detecteed by PMPS",
)
ax.plot(
    v_bins,
    pdf_kick_velocity_maxwell(v_bins) * 5e6,
    linestyle="--",
    lw=4,
    color="black",
    alpha=1,
    label=r"Maxwell $\sigma = 265$ km s$^{-1}$",
)
ax.set_xlabel(r"Total velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.yscale('log')
plt.ylim(0.1, 3.e6)
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

## Magneto-rotational information

Histogramming the periods, magnetic fields and misalignment angles

In [ ]:
def Gaussian(x, mean, sigma):
    y = (
        1
        / (sigma * np.sqrt(2 * np.pi))
        * np.exp(-((x - mean) ** 2) / (2 * sigma ** 2))
    )
    return y

In [ ]:
P_bins = np.logspace(-2., 2., 31)
P_grid = np.logspace(-2., 2., 1000)
print(max(P), min(P))

In [ ]:
pdf_P_initial_area = quad(Gaussian, 0, 100, args=(cfg["P_initial_mean"], cfg["P_initial_sigma"]))[0]
print(pdf_P_initial_area)

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    P,
    bins=P_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulated all",
)
ax.hist(
    P[intercepted],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="intercepted our LOS",
)
ax.hist(
    P[detected],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="detected by PMPS",
)
ax.plot(P_grid, 
        Gaussian(
            P_grid, 
            cfg["P_initial_mean"], 
            cfg["P_initial_sigma"]
        ) / pdf_P_initial_area * 5.e4,
        linestyle="--",
        lw=4,
        color="black",
        label="Theoretical initial",
)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
#plt.xlim(0., 50.0)
plt.ylim(0.1, 2.e5)
plt.xscale('log')
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
B_log10_bins = np.linspace(9.0, 17.0, 51)
print(max(np.log10(B)), min(np.log10(B)))

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    np.log10(B),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulation all",
)
ax.hist(
    np.log10(B[intercepted]),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="intercepting our LOS",
)
ax.hist(
    np.log10(B[detected]),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="detected by PMPS",
)
ax.plot(B_log10_bins, 
        Gaussian(
            B_log10_bins, 
            cfg["B_initial_log10_mean"], 
            cfg["B_initial_log10_sigma"]
        ) * 9.e3,
        linestyle="--",
        lw=4,
        color="black",
        label="Theoretical initial",
)
plt.xlabel(r"log$_{10} B$ [G]")
plt.ylabel(r"Number of NSs")
plt.xlim(9., 17.0)
plt.ylim(0.1, 5.e4)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
chi_bins = np.linspace(0, np.pi / 2, 51)
print(max(chi), min(chi))

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    chi,
    bins=chi_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulated all",
)
ax.hist(
    chi[intercepted],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="intercepting our LOS",
)
ax.hist(
    chi[detected],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="detected by PMPS",
)
ax.plot(chi_bins, 
        np.sin(chi_bins) * 5.e3,
        linestyle="--",
        lw=4,
        color="black",
        label="Theoretical initial",
)
plt.xlabel(r"$\chi$ [rad]")
plt.ylabel(r"Normalized PDF")
plt.xlim(0., np.pi / 2)
plt.ylim(0.1, 5.e4)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Plotting the PPdot diagram of the final pulsar population, representin a snap-shot at the current time.

In [ ]:
len(P[::40])

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    P,
    P_dot / const.YR_TO_S,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.2,
    rasterized=True
)
ax.plot(
    P[intercepted],
    P_dot[intercepted] / const.YR_TO_S,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=0.2,
    rasterized=True
)
ax.plot(
    P[detected],
    P_dot[detected] / const.YR_TO_S,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.,
    rasterized=True
)

ax.set_xscale('log')
ax.set_yscale('log')

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")

plt.show()

## Time-evolution in the PPdot diagram

In [ ]:
import pypopsyn.simulator.magneto_rotational_physics.magneto_rotational_evolution as mre
import pypopsyn.simulator.magneto_rotational_physics.period_derivative as pdv
import pypopsyn.simulator.magneto_rotational_physics.magnetic_field_derivative as mfdv
from scipy.integrate import solve_ivp

Consider two pulsars which differ only by their initial fields.

In [ ]:
cfg["NS_number"] = 2
cfg["t_age_max"] = 3e7
cfg["time_step"] = 1e1

B_initial_test = np.array([1e12, 1e14])
chi_initial_test = np.array([np.pi / 3, np.pi / 3])
P_initial_test = np.array([0.01, 0.01])
t_age_test = np.array([cfg["t_age_max"], cfg["t_age_max"]])

Determine their initial period derivatives in units of $[s/s]$.

In [ ]:
P_dot_initial_test = np.zeros(2)

P_dot_initial_test[0] = (
    pdv.period_derivative(
        B_initial_test[0], chi_initial_test[0], P_initial_test[0]
    )
    / const.YR_TO_S
)

P_dot_initial_test[1] = (
    pdv.period_derivative(
        B_initial_test[1], chi_initial_test[1], P_initial_test[1]
    )
    / const.YR_TO_S
)

Obtaint the evolution in time for a single object:

In [ ]:
def magneto_rotational_evolution_tracked_single_object(
    B_initial, chi_initial, P_initial, t_age,
):
    # Initialization of the time grid.
    time_grid = np.append(
        10 ** np.arange(0, np.log10(t_age), cfg["magrot_time_step_log10"],), t_age,
    )

    # Initial conditions for the three parameters.
    y_initial = np.array([B_initial, chi_initial, P_initial])

    evol_output = solve_ivp(
        mre.combined_derivatives,
        t_span=[1.0, t_age],
        y0=y_initial,
        method="RK45",
        t_eval=time_grid,
        args=(B_initial,),
    ).y

    return evol_output

In [ ]:
B_A, chi_A, P_A = magneto_rotational_evolution_tracked_single_object(
    B_initial_test[0], chi_initial_test[0], P_initial_test[0], t_age_test[0],
)

B_B, chi_B, P_B = magneto_rotational_evolution_tracked_single_object(
    B_initial_test[1], chi_initial_test[1], P_initial_test[1], t_age_test[1],
)

In [ ]:
period_derivative_vect = np.vectorize(pdv.period_derivative)

P_dot_A = period_derivative_vect(B_A, chi_A, P_A) / const.YR_TO_S
P_dot_B = period_derivative_vect(B_B, chi_B, P_B) / const.YR_TO_S

As a comparison, determine the period derivatives in $[s/s]$ at the current time with the package functions.

In [ ]:
B_final_test, chi_final_test, P_final_test, evol_dictionary = mre.magneto_rotational_evolution(
    B_initial_test, chi_initial_test, P_initial_test, t_age_test
)

In [ ]:
P_dot_final_test = np.zeros(2)

P_dot_final_test[0] = (
    pdv.period_derivative(B_final_test[0], chi_final_test[0], P_final_test[0])
    / const.YR_TO_S
)

P_dot_final_test[1] = (
    pdv.period_derivative(B_final_test[1], chi_final_test[1], P_final_test[1])
    / const.YR_TO_S
)

In [ ]:
fig, ax = plt.subplots()

ax.loglog(P[::40], P_dot[::40] / const.YR_TO_S, ".", color="darkgray", ms=6)
ax.loglog(P_A, P_dot_A, "-", color="tab:orange")
ax.loglog(P_B, P_dot_B, "-", color="tab:red")
ax.loglog(P_initial_test, P_dot_initial_test, "o", color="tab:blue", ms=8)
ax.loglog(P_final_test, P_dot_final_test, "o", color="tab:green", ms=8)

ax.set_xlim(5e-3, 1e2)

plt.show()

This again shows how sensitive the code is to the oldest neutron star in the sample (for a given initial period distribution).

THINGS TO THINK ABOUT: CUT-OFF AT THE BOTTOM LOOKS A BIT WEIRD, SOME OF THE PERIODS SEEM RATHER LARGE.

Plot the three quantities as a function of time.

In [ ]:
time_grid = np.append(
    10 ** np.arange(0, np.log10(t_age_test[0]), cfg["magrot_time_step_log10"],),
    t_age_test[0],
)

Magnetic field evolution timescales.

In [ ]:
timescale_ohm = mfdv.timescale_ohmic(cfg["L"], cfg["sigma"])
timescale_Hall_A = mfdv.timescale_Hall(B_initial_test[0], cfg["L"], cfg["n_e"])
timescale_Hall_B = mfdv.timescale_Hall(B_initial_test[1], cfg["L"], cfg["n_e"])

print(timescale_ohm * 1e-6, timescale_Hall_A * 1e-6, timescale_Hall_B * 1e-6)

Analytical evolution according to Aguilera et al. (2008):

In [ ]:
early_times_analytic_A = B_initial_test[0] * (
    1 + time_grid / timescale_Hall_A
) ** (-1)

early_times_analytic_B = B_initial_test[1] * (
    1 + time_grid / timescale_Hall_B
) ** (-1)

late_times_analytic_A = B_initial_test[0] * np.exp(-time_grid / timescale_ohm)

late_times_analytic_B = B_initial_test[1] * np.exp(-time_grid / timescale_ohm)

In [ ]:
fig, ax = plt.subplots()

ax.loglog(time_grid, B_A, "-", color="tab:orange", lw=3, label=r"$10^{12}$ [G]")
ax.loglog(time_grid, early_times_analytic_A, "--", color="tab:orange", lw=3)
ax.loglog(time_grid, late_times_analytic_A, "-.", color="tab:orange", lw=3)

ax.loglog(time_grid, B_B, "-", color="tab:red", lw=3, label=r"$10^{14}$ [G]")
ax.loglog(time_grid, early_times_analytic_B, "--", color="tab:red", lw=3)
ax.loglog(time_grid, late_times_analytic_B, "-.", color="tab:red", lw=3)

ax.set_xlim(1, 1e8)

plt.xlabel(r"$t$ [yr]")
plt.ylabel(r"$B$ [G]")
plt.legend(loc=3)

plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.semilogx(
    time_grid,
    chi_A * 180 / np.pi,
    "-",
    color="tab:orange",
    lw=3,
    label=r"$10^{12}$ [G]",
)
ax.semilogx(
    time_grid,
    chi_B * 180 / np.pi,
    "-",
    color="tab:red",
    lw=3,
    label=r"$10^{14}$ [G]",
)

ax.set_xlim(1, 5e7)

plt.xlabel(r"$t$ [yr]")
plt.ylabel(r"$\chi$ [deg]")
plt.legend(loc=1)

plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.loglog(time_grid, P_A, "-", color="tab:orange", lw=3, label=r"$10^{12}$ [G]")
ax.loglog(time_grid, P_B, "-", color="tab:red", lw=3, label=r"$10^{14}$ [G]")

ax.set_xlim(1, 5e7)

plt.xlabel(r"$t$ [yr]")
plt.ylabel(r"$P$ [s]")
plt.legend(loc=2)

plt.show()

Plot histogram of observed radio fluxes

In [ ]:
# Convert radio fluxes in Jy.
print(max(S_radio_Jy[intercepted]), min(S_radio_Jy[intercepted]))
S_radio_bins = np.logspace(-10, 2, 51)

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    S_radio_Jy[intercepted],
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="intercepting our LOS",
)
ax.hist(
    S_radio_Jy[detected],
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="detected by PMPS",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale('log')
ax.set_yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * cfg["NS_mass"] * cfg["NS_radius"] ** 2
Erot_dot = NS_inertia * (2.*np.pi)**2 * P_dot / const.YR_TO_S / (P ** 3)
pseudo_L_radio = S_radio_Jy * dist**2 * 1000

In [ ]:
logErot_dot = np.log10(Erot_dot[detected])
logL_radio = np.log10(pseudo_L_radio[detected])

# Compute the barycenter of the data.
xm = np.mean(logErot_dot)
ym = np.mean(logL_radio)

# fit the data with curve_fit: popt[0] and popt[1] are the optimal a and b parameters; pcov is the covariance matrix
popt, pcov = curve_fit(linear_relation, logErot_dot-xm, logL_radio-ym, [0.1, 1.e-10])

# evaluate the 1-sigma error on the parameters
perr = np.sqrt(np.diag(pcov))

a = popt[0]
b = popt[1]
a_err = perr[0]
b_err = perr[1]

x_grid = np.linspace(27.,40.,1000)
best_fit_pseudo = 10**(a*(x_grid-xm)+b+ym)

print(f"Results of the linear fit in log: \n a = {a} +- {a_err}; \n b = {b} +- {b_err}")

In [ ]:
logErot_dot = np.log10(Erot_dot[detected])
logL_radio = np.log10(L_radio[detected])

# Compute the barycenter of the data.
xm = np.mean(logErot_dot)
ym = np.mean(logL_radio)

# fit the data with curve_fit: popt[0] and popt[1] are the optimal a and b parameters; pcov is the covariance matrix
popt, pcov = curve_fit(linear_relation, logErot_dot-xm, logL_radio-ym, [0.1, 1.e-10])

# evaluate the 1-sigma error on the parameters
perr = np.sqrt(np.diag(pcov))

a = popt[0]
b = popt[1]
a_err = perr[0]
b_err = perr[1]

x_grid = np.linspace(27.,40.,1000)
best_fit_true = 10**(a*(x_grid-xm)+b+ym)

print(f"Results of the linear fit in log: \n a = {a} +- {a_err}; \n b = {b} +- {b_err}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(15,8))
ax1.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax1.set_ylabel(r"$S d^2$ [mJy kpc$^2$]")
ax1.loglog(Erot_dot[detected], pseudo_L_radio[detected], 'o', color='tab:red', ms=6, alpha=0.5, rasterized=True, label="pseudo luminosity",)
ax1.plot(10**x_grid, best_fit_pseudo, linestyle='-',linewidth=3,color='tab:red')

ax2 = ax1.twinx()
ax2.set_ylabel(r"$L(\nu)$ [erg s$^{-1}$ Hz$^{-1}$]")
ax2.loglog(Erot_dot[detected], L_radio[detected], 's', color='tab:green', ms=6, alpha=0.5, rasterized=True, label="true luminosity",)
ax2.plot(10**x_grid, best_fit_true, linestyle='-',linewidth=3,color='tab:green')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
plt.legend(lines1 + lines2, labels1 + labels2, bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)
plt.show(block=False)

In [ ]:
w_int_deg = w_int / P * 360.

In [ ]:
def fit_MG2011(P: np.ndarray) -> np.ndarray:
    """
    Fit of the W50 pulse widths of double pulse pulsars as a function of their spin period from Maciesiak and Gil 2011.
    
    Args:
        P (np.ndarray): Spin period in [s].
    
    Returns:
        (np.ndarray): value of the W50 predicted by the model in [deg].
    """
    w50 = 3.5 * P**(-0.5)
    
    return w50

w_theory = 2 * er.beam_aperture(P_grid, cfg["r_em"])/np.pi*180.

In [ ]:
fig, ax = plt.subplots(figsize=(15,8))

ax.loglog(P[intercepted], w_int_deg[intercepted], '.', color='tab:blue', ms=6, rasterized=True, label="intercepting our LOS",)
ax.loglog(P[detected], w_int_deg[detected], '.', color='tab:red', ms=6, rasterized=True,label="detected by PMPS",)
ax.loglog(P_grid, fit_MG2011(P_grid), "--", color="black", lw=3, label=r"$w_{50} = 2.5^{\circ} P^{-0.5}$")
ax.loglog(P_grid, w_theory, "-", color="black", lw=3, label=r"theoretical model")
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$w$ [rad]")

plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)
plt.show(block=False)